In [ ]:
# SMOTE + XGBoost on frozen embeddings

# Reuses the **cached frozen-encoder embeddings** written by `04_extract_embeddings`(`embeddings_detect/`). 
# Evaluates CKD vs non-CKD with **stratified 5-fold CV**,
# **in-fold SMOTE**, and **XGBoost**, reporting accuracy, recall, F1 and AUC, and
# writes `results_table.tex` for the paper.

# This is the *final* frozen pipeline; only the classifier and the imbalance handling differ from notebook 14.

# Run `04_extract_embeddings` first so `embeddings_detect/` exists.t`.gboost`.

In [3]:
!pip install imbalanced-learn xgboost  # if not already installed

LABELS    = "labels/labels_detect_any_ckd.json"   # binary CKD (1) vs non-CKD (0)
OUT_DIR   = "embeddings_detect"                    # cached by notebook 04
SEED      = 0
N_SPLITS  = 5
N_REPEATS = 1          # 1 = plain stratified 5-fold; raise to 5 to average shuffles
ID_AUC    = 0.719      # detect_any_ckd identifier baseline (rank); NOT 0.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 867.5 kB/s eta 0:00:000:0100:01


In [5]:
import json, numpy as np
from pathlib import Path
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline            # imblearn, NOT sklearn
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

lab = json.load(open(LABELS))["labels"]
pos = sum(lab.values()); neg = len(lab) - pos
print(f"CKD={pos}  no-CKD={neg}  majority-acc={max(pos,neg)/len(lab):.3f}  "
      f"(ID baseline AUC {ID_AUC})\n")

# built-in string scorers: 'roc_auc' uses predict_proba automatically
scoring = {"accuracy": "accuracy", "recall": "recall", "f1": "f1", "auc": "roc_auc"}

def evaluate(X, y):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("smote", SMOTE(k_neighbors=5, random_state=SEED)),   # TRAIN fold only
                     ("clf",   XGBClassifier(n_estimators=400, max_depth=4,
                               learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                               eval_metric="logloss", random_state=SEED, n_jobs=-1))])
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    r = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return {k: (r[f"test_{k}"].mean(), r[f"test_{k}"].std()) for k in scoring}

PRETTY = {"grayscale_heatmap":"Greyscale", "rgb_mz_split":"RGB m/z split",
          "pseudocolor_heatmap":"Pseudocolour",
          "resnet18":"ResNet-18", "efficientnet_b0":"EfficientNet-B0", "vit_b_16":"ViT-B/16"}

print(f"{'representation':18s} {'encoder':16s} {'acc':>6} {'rec':>6} {'f1':>6} {'auc':>6}")
print("-"*58)
rows = []
for f in sorted(Path(OUT_DIR).glob("*.npy")):
    meta = f.with_suffix(".json")
    if not meta.exists(): continue
    info = json.load(open(meta))
    X = np.load(f); y = np.array([lab[p] for p in info["pids"]])
    if len(np.unique(y)) < 2: continue
    m = evaluate(X, y)
    name = f"{PRETTY.get(info['representation'],info['representation'])} + {PRETTY.get(info['model'],info['model'])}"
    rows.append((name, m))
    print(f"{info['representation']:18s} {info['model']:16s} "
          f"{m['accuracy'][0]:6.3f} {m['recall'][0]:6.3f} {m['f1'][0]:6.3f} {m['auc'][0]:6.3f}")

CKD=771  no-CKD=106  majority-acc=0.879  (ID baseline AUC 0.719)

representation     encoder             acc    rec     f1    auc
----------------------------------------------------------
grayscale_heatmap  efficientnet_b0   0.805  0.910  0.889  0.664
grayscale_heatmap  resnet18          0.819  0.939  0.899  0.656
grayscale_heatmap  vit_b_16          0.799  0.914  0.886  0.633
rgb_mz_split       efficientnet_b0   0.799  0.902  0.884  0.702
rgb_mz_split       resnet18          0.834  0.951  0.907  0.661
rgb_mz_split       vit_b_16          0.797  0.912  0.885  0.649


In [11]:
import json, numpy as np
from pathlib import Path
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
from imblearn.pipeline import Pipeline           
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

SEED, N_SPLITS, N_REPEATS = 0, 5, 10 #5       

# all tasks reuse the detection embeddings (their patients are a subset)
TASKS = {
    "detect_any_ckd":       dict(labels="labels/labels_detect_any_ckd.json",      id_auc=0.719),
    "severity_all":         dict(labels="labels/labels_severity_all.json",        id_auc=0.535),
    "severity_nondiabetic": dict(labels="labels/labels_severity_nondiabetic.json", id_auc=0.519),
    "severity_diabetic":    dict(labels="labels/labels_severity_diabetic.json",   id_auc=0.632),
}
EMB_DIR    = "embeddings_detect"
RESAMPLING = ["none", "smote", "scale_pos_weight"]

scoring = {"accuracy":  "accuracy",
           "precision": make_scorer(precision_score, zero_division=0),
           "recall":    make_scorer(recall_score,    zero_division=0),
           "f1":        make_scorer(f1_score,        zero_division=0),
           "auc":       "roc_auc"}

def make_pipe(kind, y):
    kw = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
              colsample_bytree=0.8, eval_metric="logloss", random_state=SEED, n_jobs=-1)
    if kind == "scale_pos_weight":
        kw["scale_pos_weight"] = (y == 0).sum() / max((y == 1).sum(), 1)   # neg/pos
    steps = [("scale", StandardScaler())]
    if kind == "smote":
        steps.append(("smote", SMOTE(k_neighbors=5, random_state=SEED)))
    steps.append(("clf", XGBClassifier(**kw)))
    return Pipeline(steps)

def evaluate(X, y, kind):
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    r = cross_validate(make_pipe(kind, y), X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return {k: r[f"test_{k}"].mean() for k in scoring}

# load every embedding file once (shared across tasks)
EMB = []
for f in sorted(Path(EMB_DIR).glob("*.npy")):
    meta = f.with_suffix(".json")
    if meta.exists():
        info = json.load(open(meta))
        EMB.append((info["representation"], info["model"], info["pids"], np.load(f)))

for task, cfg in TASKS.items():
    lab = json.load(open(cfg["labels"]))["labels"]
    pos = sum(lab.values()); neg = len(lab) - pos
    print(f"\n### {task}   pos={pos} neg={neg} "
          f"majority={max(pos,neg)/len(lab):.3f}  base-rate(pos)={pos/len(lab):.3f}  "
          f"ID-baseline AUC={cfg['id_auc']}")
    print(f"{'rep':17s} {'encoder':16s} {'resample':16s} "
          f"{'acc':>6} {'prec':>6} {'rec':>6} {'f1':>6} {'auc':>6}")
    print("-"*84)
    for rep, model, pids, Xall in EMB:
        keep = [i for i, p in enumerate(pids) if p in lab]
        if not keep: continue
        X = Xall[keep]; y = np.array([lab[pids[i]] for i in keep])
        if len(np.unique(y)) < 2: continue
        for kind in RESAMPLING:
            m = evaluate(X, y, kind)
            print(f"{rep:17s} {model:16s} {kind:16s} "
                  f"{m['accuracy']:6.3f} {m['precision']:6.3f} {m['recall']:6.3f} "
                  f"{m['f1']:6.3f} {m['auc']:6.3f}")


### detect_any_ckd   pos=771 neg=106 majority=0.879  base-rate(pos)=0.879  ID-baseline AUC=0.719
rep               encoder          resample            acc   prec    rec     f1    auc
------------------------------------------------------------------------------------
grayscale_heatmap efficientnet_b0  none              0.846  0.857  0.984  0.916  0.674
grayscale_heatmap efficientnet_b0  smote             0.806  0.867  0.913  0.889  0.672
grayscale_heatmap efficientnet_b0  scale_pos_weight  0.819  0.863  0.937  0.898  0.672
grayscale_heatmap resnet18         none              0.851  0.858  0.989  0.919  0.653
grayscale_heatmap resnet18         smote             0.825  0.863  0.945  0.902  0.669
grayscale_heatmap resnet18         scale_pos_weight  0.828  0.863  0.949  0.904  0.648
grayscale_heatmap vit_b_16         none              0.848  0.859  0.983  0.917  0.666
grayscale_heatmap vit_b_16         smote             0.811  0.868  0.919  0.892  0.657
grayscale_heatmap vit_b_16        